# Fundamentos de datos: CSV, NumPy y Pandas

Este notebook es el **paso previo** al análisis del Titanic (`titanic_datos_limpieza_narrativo.ipynb`).

Aquí aprenderás las piezas básicas que necesitas antes de limpiar datos reales:

- qué es un **CSV** y cómo crearlo,
- cómo usar **NumPy** para trabajar con números en bloque,
- cómo usar **Pandas** para manejar tablas como en Excel, pero con código.

No necesitas saber programación avanzada. Solo necesitas ir celda por celda, ejecutar y leer la explicación.


In [5]:
!pip install pandas numpy

## 0. ¿Qué vamos a necesitar?

Al final de este notebook deberías poder:

1. Crear y guardar un archivo `.csv`.
2. Cargar ese archivo con `pandas.read_csv()`.
3. Entender qué es un `array` de NumPy.
4. Explorar una tabla con `head()`, `shape`, `describe()`.
5. Filtrar filas, seleccionar columnas y detectar valores faltantes.
6. Agrupar datos y crear columnas nuevas.

Todo eso lo usarás después en el notebook del Titanic.


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

def resolver_carpeta_datos() -> Path:
    """Encuentra o crea una carpeta data escribible dentro del proyecto."""
    cwd = Path.cwd()

    for raiz in [cwd, cwd.parent]:
        if raiz == cwd.parent and raiz == cwd:
            continue
        data_dir = raiz / "data"
        if data_dir.exists() or (raiz / "README.md").exists():
            data_dir.mkdir(parents=True, exist_ok=True)
            return data_dir

    for base in [cwd, Path.home() / "datos_curso"]:
        data_dir = base / "data"
        try:
            data_dir.mkdir(parents=True, exist_ok=True)
            return data_dir
        except PermissionError:
            continue

    raise PermissionError("No se pudo crear una carpeta data escribible.")

DATA_DIR = resolver_carpeta_datos()
REPO_ROOT = DATA_DIR.parent

print("Carpeta de datos:", DATA_DIR)


Carpeta de datos: /home/ubuntu/data


## 1. ¿Qué es un CSV?

**CSV** significa *Comma Separated Values* (valores separados por comas).

Es un archivo de texto plano que guarda una tabla. Cada **fila** es un registro y cada **columna** está separada por comas.

Ejemplo visual:

```text
nombre,edad,ciudad
Ana,28,Puebla
Luis,31,CDMX
María,24,Mérida
```

Ventajas del CSV:

- casi cualquier programa lo abre (Excel, Google Sheets, Python),
- es ligero y fácil de compartir,
- es el formato más común en cursos y proyectos de datos.

En datos reales verás archivos como `titanic.csv`, `ventas.csv` o `alumnos.csv`.


### 1.1 Crear un CSV a mano (idea conceptual)

Podrías crear un archivo con un editor de texto y guardarlo como `ejemplo.csv`.

Pero en análisis de datos es más común **generar el CSV desde Python**, porque así puedes repetir el proceso, corregir errores y escalar a miles de filas.


In [7]:
# Opción A: escribir un CSV línea por línea con Python puro.
csv_path_manual = DATA_DIR / "ejemplo_manual.csv"

lineas = [
    "nombre,edad,ciudad",
    "Ana,28,Puebla",
    "Luis,31,CDMX",
    "María,24,Mérida",
]

with open(csv_path_manual, "w", encoding="utf-8") as archivo:
    archivo.write("\n".join(lineas))

print("Archivo creado:", csv_path_manual)

# Ver el contenido como texto.
print("\nContenido del archivo:")
print(csv_path_manual.read_text(encoding="utf-8"))


Archivo creado: /home/ubuntu/data/ejemplo_manual.csv

Contenido del archivo:
nombre,edad,ciudad
Ana,28,Puebla
Luis,31,CDMX
María,24,Mérida


### 1.2 Crear un CSV con Pandas (forma recomendada)

En la práctica usamos **Pandas** porque convierte listas y diccionarios en tablas de forma sencilla.

Flujo típico:

1. Tener los datos en Python (listas o diccionarios).
2. Crear un `DataFrame`.
3. Guardarlo con `.to_csv()`.


In [8]:
# Datos en forma de diccionario: cada llave es una columna.
datos_alumnos = {
    "matricula": ["A001", "A002", "A003", "A004", "A005"],
    "nombre": ["Ana", "Luis", "María", "Jorge", "Sofía"],
    "edad": [20, 22, 21, 23, 20],
    "carrera": ["Datos", "Finanzas", "Datos", "Mercadotecnia", "Datos"],
    "promedio": [8.7, 7.9, 9.1, 6.8, 8.4],
}

# DataFrame = tabla de Pandas.
df_alumnos = pd.DataFrame(datos_alumnos)

print("Tabla creada en memoria:")
display(df_alumnos)

# Guardar como CSV.
csv_alumnos = DATA_DIR / "alumnos_mini.csv"
df_alumnos.to_csv(csv_alumnos, index=False)

print("\nCSV guardado en:", csv_alumnos)


Tabla creada en memoria:


,matricula,nombre,edad,carrera,promedio
0,A001,Ana,20,Datos,8.7
1,A002,Luis,22,Finanzas,7.9
2,A003,María,21,Datos,9.1
3,A004,Jorge,23,Mercadotecnia,6.8
4,A005,Sofía,20,Datos,8.4



CSV guardado en: /home/ubuntu/data/alumnos_mini.csv


**Nota sobre `index=False`:**

Pandas, por defecto, puede guardar una columna extra con el número de fila (índice). En la mayoría de los casos **no la queremos** en el CSV final, por eso usamos `index=False`.


## 2. Leer un CSV con Pandas

Leer un CSV es lo contrario de guardarlo: Python toma el archivo y lo convierte en un `DataFrame` para trabajar en memoria.

Función clave: `pd.read_csv(ruta_del_archivo)`


In [9]:
# Leemos el archivo que acabamos de crear.
df = pd.read_csv(csv_alumnos)

print("Tipo de objeto:", type(df))
print("Dimensiones (filas, columnas):", df.shape)

df


Tipo de objeto: <class 'pandas.DataFrame'>
Dimensiones (filas, columnas): (5, 5)


,matricula,nombre,edad,carrera,promedio
0,A001,Ana,20,Datos,8.7
1,A002,Luis,22,Finanzas,7.9
2,A003,María,21,Datos,9.1
3,A004,Jorge,23,Mercadotecnia,6.8
4,A005,Sofía,20,Datos,8.4


### Primeras preguntas al cargar datos

Siempre conviene revisar:

- `df.head()` → primeras filas
- `df.tail()` → últimas filas
- `df.columns` → nombres de columnas
- `df.dtypes` → tipo de cada columna
- `df.info()` → resumen general


In [10]:
print("Primeras filas:")
display(df.head(3))

print("\nÚltimas filas:")
display(df.tail(2))

print("\nNombres de columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
display(df.dtypes.to_frame("tipo"))

print("\nResumen con info():")
df.info()


Primeras filas:


,matricula,nombre,edad,carrera,promedio
0,A001,Ana,20,Datos,8.7
1,A002,Luis,22,Finanzas,7.9
2,A003,María,21,Datos,9.1



Últimas filas:


,matricula,nombre,edad,carrera,promedio
3,A004,Jorge,23,Mercadotecnia,6.8
4,A005,Sofía,20,Datos,8.4



Nombres de columnas:
['matricula', 'nombre', 'edad', 'carrera', 'promedio']

Tipos de datos:


,tipo
matricula,str
nombre,str
edad,int64
carrera,str
promedio,float64



Resumen con info():
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   matricula  5 non-null      str    
 1   nombre     5 non-null      str    
 2   edad       5 non-null      int64  
 3   carrera    5 non-null      str    
 4   promedio   5 non-null      float64
dtypes: float64(1), int64(1), str(3)
memory usage: 332.0 bytes


## 3. NumPy: números en bloque

**NumPy** (`numpy`) es la librería base para trabajar con arreglos numéricos en Python.

Pandas está construido sobre NumPy. Por eso conviene entender la idea de un **array** antes de seguir.

Un `array` es como una lista de números, pero optimizada para cálculos matemáticos.


In [11]:
# Lista normal de Python.
lista_edades = [20, 22, 21, 23, 20]
print("Lista de Python:", lista_edades)

# Array de NumPy.
array_edades = np.array(lista_edades)
print("Array de NumPy:", array_edades)
print("Tipo:", type(array_edades))

# Operaciones con todo el array a la vez.
print("\nEdades + 1:", array_edades + 1)
print("Promedio:", array_edades.mean())
print("Edad mínima:", array_edades.min())
print("Edad máxima:", array_edades.max())


Lista de Python: [20, 22, 21, 23, 20]
Array de NumPy: [20 22 21 23 20]
Tipo: <class 'numpy.ndarray'>

Edades + 1: [21 23 22 24 21]
Promedio: 21.2
Edad mínima: 20
Edad máxima: 23


### Arrays 2D: pensar en filas y columnas

Un array bidimensional se parece a una mini tabla:

```text
[[20, 22],
 [21, 23],
 [20, 19]]
```

Esto ayuda a entender que una tabla es, en el fondo, una estructura numérica organizada.


In [12]:
matriz = np.array([
    [20, 22],
    [21, 23],
    [20, 19],
])

print("Matriz:\n", matriz)
print("Forma (filas, columnas):", matriz.shape)
print("Promedio por columna:", matriz.mean(axis=0))
print("Promedio por fila:", matriz.mean(axis=1))


Matriz:
 [[20 22]
 [21 23]
 [20 19]]
Forma (filas, columnas): (3, 2)
Promedio por columna: [20.33333333 21.33333333]
Promedio por fila: [21.  22.  19.5]


### ¿Por qué NumPy si ya tengo Pandas?

- NumPy es rápido para operaciones matemáticas.
- Pandas usa NumPy por debajo en las columnas numéricas.
- Muchas funciones de limpieza y modelos estadísticos esperan arrays numéricos.

En el notebook del Titanic verás cosas como `.values`, `.mean()`, comparaciones numéricas y correlaciones. Todo eso se apoya en NumPy.


## 4. Explorar una tabla con Pandas

Una vez cargado el CSV, exploramos para entender qué tenemos.

Herramientas básicas:

| Función | Para qué sirve |
|---------|----------------|
| `head()` | Ver el inicio de la tabla |
| `shape` | Contar filas y columnas |
| `describe()` | Resumen estadístico de columnas numéricas |
| `value_counts()` | Contar categorías |
| `unique()` | Ver valores distintos |


In [13]:
print("--- Exploración básica ---")

print("Filas y columnas:", df.shape)
print("\nResumen numérico:")
display(df.describe())

print("\nCuántos alumnos hay por carrera:")
display(df["carrera"].value_counts())

print("\nValores únicos de carrera:")
print(df["carrera"].unique())


--- Exploración básica ---
Filas y columnas: (5, 5)

Resumen numérico:


,edad,promedio
count,5.00000,5.00000
mean,21.20000,8.18000
std,1.30384,0.88713
min,20.00000,6.80000
25%,20.00000,7.90000
50%,21.00000,8.40000
75%,22.00000,8.70000
max,23.00000,9.10000



Cuántos alumnos hay por carrera:


carrera
Datos            3
Finanzas         1
Mercadotecnia    1
Name: count, dtype: int64


Valores únicos de carrera:
<StringArray>
['Datos', 'Finanzas', 'Mercadotecnia']
Length: 3, dtype: str


## 5. Seleccionar columnas y filas

### Seleccionar columnas

- Una columna → `df["nombre"]` devuelve una **Series**.
- Varias columnas → `df[["nombre", "promedio"]]` devuelve un **DataFrame**.

### Seleccionar filas por posición

- `df.iloc[0]` → primera fila
- `df.iloc[0:3]` → primeras 3 filas

### Seleccionar por etiqueta

- `df.loc[0, "nombre"]` → fila 0, columna nombre


In [14]:
# Una columna (Series).
print("Columna nombre:")
display(df["nombre"])

# Varias columnas (DataFrame).
print("\nSubtabla nombre + promedio:")
display(df[["nombre", "promedio"]])

# Primera fila por posición.
print("\nPrimera fila:")
display(df.iloc[0:1])

# Valor específico.
print("\nNombre de la fila 2:", df.loc[2, "nombre"])


Columna nombre:


0      Ana
1     Luis
2    María
3    Jorge
4    Sofía
Name: nombre, dtype: str


Subtabla nombre + promedio:


,nombre,promedio
0,Ana,8.7
1,Luis,7.9
2,María,9.1
3,Jorge,6.8
4,Sofía,8.4



Primera fila:


,matricula,nombre,edad,carrera,promedio
0,A001,Ana,20,Datos,8.7



Nombre de la fila 2: María


## 6. Filtrar datos (muy importante)

Filtrar significa quedarnos solo con las filas que cumplen una condición.

Ejemplo:

> Quiero ver solo alumnos de la carrera de Datos con promedio mayor a 8.

Esto se hace con una **máscara booleana**: una serie de `True` y `False` que indica qué filas conservar.


In [15]:
# Condición: carrera == "Datos"
condicion_carrera = df["carrera"] == "Datos"
print("Máscara de carrera Datos:")
display(condicion_carrera)

# Aplicar filtro.
df_datos = df[condicion_carrera]
print("\nSolo carrera Datos:")
display(df_datos)

# Combinar dos condiciones con & (y) y | (o).
filtro = (df["carrera"] == "Datos") & (df["promedio"] > 8)
print("\nDatos con promedio > 8:")
display(df[ filtro ])


Máscara de carrera Datos:


0     True
1    False
2     True
3    False
4     True
Name: carrera, dtype: bool


Solo carrera Datos:


,matricula,nombre,edad,carrera,promedio
0,A001,Ana,20,Datos,8.7
2,A003,María,21,Datos,9.1
4,A005,Sofía,20,Datos,8.4



Datos con promedio > 8:


,matricula,nombre,edad,carrera,promedio
0,A001,Ana,20,Datos,8.7
2,A003,María,21,Datos,9.1
4,A005,Sofía,20,Datos,8.4


## 7. Valores faltantes

En datos reales **casi siempre** hay información incompleta. En el Titanic faltan edades y cabinas.

Funciones clave:

- `df.isnull()` → detecta nulos
- `df.isnull().sum()` → cuenta nulos por columna
- `df.dropna()` → elimina filas con nulos
- `df.fillna(valor)` → reemplaza nulos

Vamos a simular un dataset con huecos para practicar.


In [16]:
df_incompleto = df.copy()

# Simulamos datos faltantes.
df_incompleto.loc[1, "promedio"] = np.nan
df_incompleto.loc[3, "edad"] = np.nan

print("Tabla con valores faltantes:")
display(df_incompleto)

print("\nConteo de nulos:")
display(df_incompleto.isnull().sum())

# Rellenar promedio faltante con la mediana.
mediana_promedio = df_incompleto["promedio"].median()
df_incompleto["promedio"] = df_incompleto["promedio"].fillna(mediana_promedio)

# Rellenar edad faltante con la mediana por carrera.
df_incompleto["edad"] = df_incompleto.groupby("carrera")["edad"].transform(
    lambda x: x.fillna(x.median())
)

print("\nDespués de imputar:")
display(df_incompleto)
print("\nNulos restantes:")
print(df_incompleto.isnull().sum().sum())


Tabla con valores faltantes:


,matricula,nombre,edad,carrera,promedio
0,A001,Ana,20.0,Datos,8.7
1,A002,Luis,22.0,Finanzas,NaN
2,A003,María,21.0,Datos,9.1
3,A004,Jorge,NaN,Mercadotecnia,6.8
4,A005,Sofía,20.0,Datos,8.4



Conteo de nulos:


matricula    0
nombre       0
edad         1
carrera      0
promedio     1
dtype: int64


Después de imputar:


,matricula,nombre,edad,carrera,promedio
0,A001,Ana,20.0,Datos,8.70
1,A002,Luis,22.0,Finanzas,8.55
2,A003,María,21.0,Datos,9.10
3,A004,Jorge,NaN,Mercadotecnia,6.80
4,A005,Sofía,20.0,Datos,8.40



Nulos restantes:
1


**Idea clave:** en el notebook del Titanic harás algo parecido con la edad: en lugar de borrar pasajeros sin edad, rellenarás con la mediana según clase y sexo.


## 8. Agrupar datos con `groupby`

`groupby` sirve para resumir información por categorías.

Ejemplos:

- promedio de calificaciones por carrera,
- número de pasajeros por clase,
- edad promedio por sexo.

Sintaxis general: `df.groupby("columna_categoria")["columna_numérica"].agg(función)`


In [17]:
print("Promedio de calificaciones por carrera:")
display(df.groupby("carrera")["promedio"].mean().round(2))

print("\nEdad promedio por carrera:")
display(df.groupby("carrera")["edad"].mean())

print("\nConteo de alumnos por carrera:")
display(df.groupby("carrera").size())

# Varias métricas a la vez.
print("\nResumen por carrera:")
display(df.groupby("carrera").agg({
    "edad": "mean",
    "promedio": ["mean", "min", "max"],
    "matricula": "count",
}))


Promedio de calificaciones por carrera:


carrera
Datos            8.73
Finanzas         7.90
Mercadotecnia    6.80
Name: promedio, dtype: float64


Edad promedio por carrera:


carrera
Datos            20.333333
Finanzas         22.000000
Mercadotecnia    23.000000
Name: edad, dtype: float64


Conteo de alumnos por carrera:


carrera
Datos            3
Finanzas         1
Mercadotecnia    1
dtype: int64


Resumen por carrera:


edad  promedio           matricula
                    mean      mean  min  max     count
carrera                                               
Datos          20.333333  8.733333  8.4  9.1         3
Finanzas       22.000000  7.900000  7.9  7.9         1
Mercadotecnia  23.000000  6.800000  6.8  6.8         1

## 9. Crear columnas nuevas

Muchas veces necesitas calcular información que no viene en el CSV original.

Ejemplos típicos:

- categorizar un número en rangos,
- marcar si alguien cumple una condición (`aprobado`, `riesgo`, `es_mayor`),
- combinar columnas.

En el Titanic crearás variables como `family_size`, `is_child` y `has_cabin`.


In [18]:
df_nuevo = df.copy()

# Columna binaria: ¿aprobado?
df_nuevo["aprobado"] = (df_nuevo["promedio"] >= 7.0).astype(int)

# Columna de categoría por rango de edad.
df_nuevo["grupo_edad"] = pd.cut(
    df_nuevo["edad"],
    bins=[0, 21, 25, 100],
    labels=["joven", "adulto_joven", "adulto"],
    right=True,
)

# Columna calculada: puntos = promedio * 10
df_nuevo["puntos"] = df_nuevo["promedio"] * 10

print("Tabla con columnas nuevas:")
display(df_nuevo)

print("\nAprobados vs no aprobados:")
display(df_nuevo["aprobado"].value_counts())


Tabla con columnas nuevas:


,matricula,nombre,edad,carrera,promedio,aprobado,grupo_edad,puntos
0,A001,Ana,20,Datos,8.7,1,joven,87.0
1,A002,Luis,22,Finanzas,7.9,1,adulto_joven,79.0
2,A003,María,21,Datos,9.1,1,joven,91.0
3,A004,Jorge,23,Mercadotecnia,6.8,0,adulto_joven,68.0
4,A005,Sofía,20,Datos,8.4,1,joven,84.0



Aprobados vs no aprobados:


aprobado
1    4
0    1
Name: count, dtype: int64

## 10. Ordenar y guardar resultados

Después de limpiar o transformar datos, conviene:

1. ordenar para revisar mejor,
2. guardar el resultado en un nuevo CSV.

Así documentas tu trabajo y puedes compartir la versión limpia.


In [19]:
# Ordenar por promedio de mayor a menor.
df_ordenado = df_nuevo.sort_values("promedio", ascending=False)
display(df_ordenado)

# Guardar versión procesada.
csv_limpio = DATA_DIR / "alumnos_mini_limpio.csv"
df_ordenado.to_csv(csv_limpio, index=False)

print("Archivo limpio guardado en:", csv_limpio)


,matricula,nombre,edad,carrera,promedio,aprobado,grupo_edad,puntos
2,A003,María,21,Datos,9.1,1,joven,91.0
0,A001,Ana,20,Datos,8.7,1,joven,87.0
4,A005,Sofía,20,Datos,8.4,1,joven,84.0
1,A002,Luis,22,Finanzas,7.9,1,adulto_joven,79.0
3,A004,Jorge,23,Mercadotecnia,6.8,0,adulto_joven,68.0


Archivo limpio guardado en: /home/ubuntu/data/alumnos_mini_limpio.csv


## 11. Mini práctica: conecta todo

Vamos a cargar el CSV limpio y responder preguntas sencillas **sin ver la tabla completa**, solo usando código.

Preguntas:

1. ¿Cuántos alumnos hay en total?
2. ¿Cuál es el promedio general?
3. ¿Quiénes tienen promedio mayor o igual a 8.5?
4. ¿Cuál es la carrera con mejor promedio?
